# 01_EDA: 基本検証ノートブック

このノートブックでは、データの基本的な性質を確認し、モデル設計の前提を検証します。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller
import sys
import os

# src読み込み用パス設定
sys.path.append('..')
from src.processing import load_and_clean_data
from src.features import generate_features
from src.pooling import pool_boj_data

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)

## 1. データ読み込みとパイプライン実行

In [ ]:
excel_path = '../data/BOJ_data.xlsx'
meeting_path = '../data/BOJ_meeting_history.csv'

df_clean = load_and_clean_data(excel_path, meeting_path)
df_feat = generate_features(df_clean)
df_pooled = pool_boj_data(df_feat)

print(f"Raw Cleaned Shape: {df_clean.shape}")
print(f"Featured Shape: {df_feat.shape}")
print(f"Pooled Shape: {df_pooled.shape}")
df_pooled.head()

## 2. M{n}スプレッドの時系列プロット

MPM日付（縦線）との関係を確認します。

In [ ]:
plt.figure(figsize=(15, 7))
for i in range(1, 9):
    col = f'M{i}_spread'
    plt.plot(df_feat['Date'], df_feat[col], label=col, alpha=0.7)

meeting_dates = df_feat[df_feat['Is_Meeting_Day'] == 1]['Date']
for d in meeting_dates:
    plt.axvline(d, color='gray', linestyle='--', alpha=0.3)

plt.title("BOJ OIS Spread (Mn - Policy Rate)")
plt.legend(ncol=4)
plt.grid(True)
plt.show()

## 3. 分数階差の確認と定常性検証

M1_spread と M1_frac_diff を比較し、ADF検定を行います。

In [ ]:
fig, ax1 = plt.subplots(figsize=(15, 6))

ax2 = ax1.twinx()
ax1.plot(df_feat['Date'], df_feat['M1_spread'], 'g-', label='M1_spread', alpha=0.5)
ax2.plot(df_feat['Date'], df_feat['M1_frac_diff'], 'b-', label='M1_frac_diff')

ax1.set_ylabel('Spread', color='g')
ax2.set_ylabel('Frac Diff', color='b')
plt.title("M1 Spread vs Fractional Difference (d=0.4)")
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')
plt.show()

print("--- ADF Test Results (p-values) ---")
for i in range(1, 9):
    col = f'M{i}_frac_diff'
    series = df_feat[col].dropna()
    if len(series) > 0:
        p_val = adfuller(series)[1]
        print(f"{col}: {p_val:.4f}")

## 4. 目的変数の分布と統計量

is_post_mpm 除外後の統計量を確認します。

In [ ]:
df_valid = df_pooled[df_pooled['is_post_mpm'] == 0].copy()
target_cols = ['Target_1d', 'Target_3d', 'Target_5d']

print(f"有効サンプル数 (is_post_mpm 除外後): {len(df_valid)}")
print("\n--- Target Statistics ---")
display(df_valid[target_cols].describe())

plt.figure(figsize=(15, 5))
for i, col in enumerate(target_cols):
    plt.subplot(1, 3, i+1)
    sns.histplot(df_valid[col].dropna(), kde=True)
    plt.title(col)
plt.tight_layout()
plt.show()

## 5. 欠損・補完状況の確認

In [ ]:
impute_cols = [f'M{i}_is_imputed' for i in range(1, 9)]
plt.figure(figsize=(15, 4))
sns.heatmap(df_feat.set_index('Date')[impute_cols].T, cmap='YlGnBu', cbar=False)
plt.title("Missing Data (Imputed) Timeline")
plt.show()